# Step 12A — policy max-norm 5 vs 20 极短对照（6/20）

本 Notebook 只比较两种 policy gradient clipping 强度的训练动力学。A/B 使用相同 seed、相同新 policy 初始化、相同 F/D/E/L、相同 N=1/2 exact Z、相同 historical median，并用人工批准的高方差工程估计覆盖 N=17/18。它不恢复旧模型/optimizer/scheduler，也不按 Reward 或 IC 选择参数。

### 第 0 格：检查安全开关、输入与对照合同

这一格不加载真实数据、不训练，通常几秒内完成。先确认 `RUN_MODE='new'`、CUDA、源目录、目标 16 次成功 update、logical hard cap=32，以及唯一变量 `policy_max_norm=5/20`。确认后把 `RUN_REAL_CLIP_COMPARISON=False` 改为 `True`。

In [ ]:
import hashlib, json, math, sys, torch
from dataclasses import asdict, replace
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd

root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'factor_gfn').is_dir())
if str(root) not in sys.path: sys.path.insert(0, str(root))

from factor_gfn.gfn import (
    ExhaustiveRegistry, GFNTrainer, NoAnchorCalibrationConfig,
    RealRewardDataPaths, RealRewardProvider, TrainingConfig,
    build_formal_stage5_no_anchor_6_20_config, build_real_reward_data_context,
    policy_state_fingerprint,
)
from factor_gfn.gfn.diagnostic_support import (
    PhaseTrackingRewardProvider, configure_registry_once, progress_heartbeat,
    run_training_with_progress,
)

RUN_REAL_CLIP_COMPARISON = False
RUN_MODE = 'new'  # 中断后改为 'resume'，从各分支最新 checkpoint 继续
DEVICE = 'cuda:0'
SEED = 42
TARGET_SUCCESSFUL_UPDATES = 16
MAX_LOGICAL_BATCHES_PER_BRANCH = 32
SOURCE_DIAGNOSTIC_ROOT = root / 'runs' / 'complexity_diagnostic_6_20' / 'manual_diagnostic_6_20_seed42'
SOURCE_REGISTRY = SOURCE_DIAGNOSTIC_ROOT / 'exhaustive_registry.sqlite3'
TARGETED_ARTIFACT = root / 'runs' / 'targeted_calibration_6_20' / 'targeted_logz_n17_n18_seed42' / 'targeted_log_z_engineering_initialization.json'
RUN_ROOT = root / 'runs' / 'policy_clip_comparison_6_20' / 'clip5_vs_clip20_seed42'
CLIP_VALUES = (5.0, 20.0)

def make_config(clip_norm):
    training = TrainingConfig(batch_size=8, learning_rate=1e-4, log_z_learning_rate=1e-2, max_steps=MAX_LOGICAL_BATCHES_PER_BRANCH, model_gradient_clip_norm=clip_norm, log_z_gradient_clip_norm=5.0, seed=SEED)
    base = build_formal_stage5_no_anchor_6_20_config(training=training)
    return replace(base, complexity=replace(base.complexity, exact_node_retry_budget=3), calibration=NoAnchorCalibrationConfig(enabled=True, target_node_counts=(17, 18)))

configs = {clip: make_config(clip) for clip in CLIP_VALUES}
print({'enabled': RUN_REAL_CLIP_COMPARISON, 'run_mode': RUN_MODE, 'device': DEVICE, 'run_root': str(RUN_ROOT), 'target_successful_updates': TARGET_SUCCESSFUL_UPDATES, 'hard_cap_logical_batches_per_branch': MAX_LOGICAL_BATCHES_PER_BRANCH, 'policy_max_norms': CLIP_VALUES, 'retry_budget': 3, 'batch_size': 8, 'policy_lr': 1e-4, 'logZ_lr': 1e-2, 'logZ_max_norm': 5.0, 'targeted_artifact': str(TARGETED_ARTIFACT)}, flush=True)
print('预计每个分支约 20–35 分钟，两个分支合计常见约 40–70 分钟；每个 logical batch 有开始/结束输出，batch 内每20秒有 heartbeat。', flush=True)

### 第 1 格：加载 training-only 数据并建立 A/B 新初始状态

这一格加载一次真实 training-only context，随后为 clip=5 和 clip=20 分别创建全新的 Trainer。每个分支只读复用 registry/exact Z/historical medians，并导入 N=17/18 的 `high_variance_engineering_estimate`。它会核对两边初始 policy、logZ 和 scheduler 完全相同；常见耗时数分钟，超过20秒会持续输出 heartbeat。

In [ ]:
if not RUN_REAL_CLIP_COMPARISON: raise RuntimeError('安全停止：确认第0格后，将 RUN_REAL_CLIP_COMPARISON=True')
if RUN_MODE not in {'new', 'resume'}: raise ValueError("RUN_MODE 只能是 'new' 或 'resume'")
if not DEVICE.startswith('cuda:') or not torch.cuda.is_available(): raise RuntimeError('本对照必须显式使用 CUDA，不回落 CPU')
for required in (SOURCE_REGISTRY, TARGETED_ARTIFACT, SOURCE_DIAGNOSTIC_ROOT / 'diagnostic_summary.json', SOURCE_DIAGNOSTIC_ROOT / 'diagnostic_context.json'):
    if not required.is_file(): raise FileNotFoundError(required)
device = torch.device(DEVICE); torch.cuda.set_device(device); torch.cuda.reset_peak_memory_stats(device)
if RUN_MODE == 'new': RUN_ROOT.mkdir(parents=True, exist_ok=False)
elif not RUN_ROOT.is_dir(): raise FileNotFoundError('resume 要求既有同名对照目录')
print('[provider] loading shared training-only data context; heartbeat every 20s', flush=True)
with progress_heartbeat('clip comparison provider context load', interval_seconds=20.0):
    data_context = build_real_reward_data_context(paths=RealRewardDataPaths())

def state_digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=True, sort_keys=True, default=list).encode('utf-8')).hexdigest()

def build_branch(clip_norm):
    label = f'clip{int(clip_norm)}'; branch_root = RUN_ROOT / label; checkpoint = branch_root / 'latest_no_anchor.pt'
    if RUN_MODE == 'new': branch_root.mkdir(parents=False, exist_ok=False)
    elif not branch_root.is_dir(): raise FileNotFoundError(branch_root)
    config = configs[clip_norm]
    base_provider = RealRewardProvider(data_context, config.reward)
    provider = PhaseTrackingRewardProvider(base_provider, audit_path=branch_root / 'reward_phase_audit.jsonl')
    manifest = provider.manifest()
    assert manifest['data_scope'] == 'training_only' and manifest['validation_oos_loaded'] is False
    registry = ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True)
    trainer = GFNTrainer(config, provider, device=device)
    with progress_heartbeat(f'{label} registry equivalence proof', interval_seconds=20.0): configure_registry_once(trainer, registry)
    if RUN_MODE == 'resume':
        if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
        trainer.load_checkpoint(checkpoint)
        print(f'[{label}] resumed step={trainer.step}, successful_updates={trainer.optimizer_step}', flush=True)
        initial_manifest = json.loads((branch_root / 'initial_state.json').read_text(encoding='utf-8'))
    else:
        with progress_heartbeat(f'{label} historical initialization proof', interval_seconds=20.0): historical = trainer.initialize_verified_historical_log_z(SOURCE_DIAGNOSTIC_ROOT)
        targeted = trainer.initialize_verified_targeted_log_z(TARGETED_ARTIFACT)
        assert targeted.initialization_status == 'high_variance_engineering_estimate' and targeted.strict_stability_check == 'failed'
        initial_log_z = {**{n: float(trainer.tb_loss.exact_tb_log_z_by_node_count[n-1]) for n in trainer.resolved_exhaustive_node_counts}, **{n: float(trainer.tb_loss.log_z_by_node_count[n-1]) for n in trainer.resolved_learned_node_counts}}
        initial_manifest = {'label': label, 'policy_state_fingerprint': policy_state_fingerprint(trainer), 'scheduler_state_fingerprint': state_digest(trainer.complexity_scheduler.state_dict()), 'initial_log_z_by_N': initial_log_z, 'targeted_provenance_fingerprint': targeted.provenance_fingerprint, 'targeted_status': targeted.initialization_status, 'strict_stability_check': targeted.strict_stability_check, 'N17_valid': targeted.calibration_statistics_by_N[17]['calibration_valid'], 'N18_valid': targeted.calibration_statistics_by_N[18]['calibration_valid'], 'N17_iqr': targeted.calibration_statistics_by_N[17]['iqr'], 'N18_iqr': targeted.calibration_statistics_by_N[18]['iqr']}
        (branch_root / 'initial_state.json').write_text(json.dumps(initial_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
        trainer.save_checkpoint(checkpoint)
    return {'label': label, 'root': branch_root, 'checkpoint': checkpoint, 'config': config, 'provider': provider, 'registry': registry, 'trainer': trainer, 'initial': initial_manifest}

branches = {clip: build_branch(clip) for clip in CLIP_VALUES}
left, right = (branches[value]['initial'] for value in CLIP_VALUES)
for key in ('policy_state_fingerprint', 'scheduler_state_fingerprint', 'initial_log_z_by_N', 'targeted_provenance_fingerprint'):
    if left[key] != right[key]: raise RuntimeError(f'A/B initial state mismatch: {key}')
print('[initialization] A/B initial policy, scheduler, logZ and targeted provenance are identical', flush=True)
print('[initialization] N17/N18', {17: left['initial_log_z_by_N']['17'] if '17' in left['initial_log_z_by_N'] else left['initial_log_z_by_N'][17], 18: left['initial_log_z_by_N']['18'] if '18' in left['initial_log_z_by_N'] else left['initial_log_z_by_N'][18]}, flush=True)

### 第 2 格：依次运行 clip=5 与 clip=20

这是唯一的长计算格。每个分支以“16次成功 optimizer update”为目标，而不是固定把 skipped batch 当成功；最多32个 logical batch。每个 batch 都保存 checkpoint，输出计数、loss、TB RMS、retry、耗时和 ETA，内部每20秒 heartbeat。中断后把第0格 `RUN_MODE` 改为 `resume` 并从头顺序运行，已完成的分支不会重训。

In [ ]:
for clip_norm in CLIP_VALUES:
    branch = branches[clip_norm]; trainer = branch['trainer']; provider = branch['provider']
    trainer.load_checkpoint(branch['checkpoint'])  # 恢复该分支自己的 RNG；避免另一分支改动全局 RNG
    print(f"[{branch['label']}] target successful updates={TARGET_SUCCESSFUL_UPDATES}; current={trainer.optimizer_step}; logical={trainer.step}/{MAX_LOGICAL_BATCHES_PER_BRANCH}", flush=True)
    while trainer.optimizer_step < TARGET_SUCCESSFUL_UPDATES and trainer.step < MAX_LOGICAL_BATCHES_PER_BRANCH:
        remaining_success = TARGET_SUCCESSFUL_UPDATES - trainer.optimizer_step
        print(f"[{branch['label']}] remaining_success={remaining_success}, next_logical={trainer.step+1}, hard_cap_remaining={MAX_LOGICAL_BATCHES_PER_BRANCH-trainer.step}", flush=True)
        with provider.phase('clip_comparison_discovery'):
            run_training_with_progress(trainer, logical_batches=1, checkpoint_path=branch['checkpoint'], checkpoint_every=1, training_audit_path=branch['root'] / 'training_health.jsonl', trajectory_audit_path=branch['root'] / 'trajectory_tb_diagnostics.jsonl')
    if trainer.optimizer_step < TARGET_SUCCESSFUL_UPDATES:
        raise RuntimeError(f"{branch['label']} reached logical hard cap with only {trainer.optimizer_step} successful updates; evidence is insufficient")
    print(f"[{branch['label']}] COMPLETE successful={trainer.optimizer_step}, logical={trainer.step}, skipped={trainer.step-trainer.optimizer_step}", flush=True)

### 第 3 格：汇总训练动力学，不自动选胜者

这一格只读取刚才的审计文件，通常几秒内完成。输出 pre-clip gradient、clip coefficient/frequency、实际参数更新、TB RMS/loss、policy entropy、skip rate、吞吐、显存，以及按 N 的 valid/exposure/TB delta。结论保持 `manual_review_required`，不会根据 Reward/IC 自动选 max-norm，也不会创建正式训练 run。

In [ ]:
def finite(values): return [float(v) for v in values if v is not None and math.isfinite(float(v))]
def median(values):
    values = finite(values); return float(np.median(values)) if values else None
def phase_mean(values, side):
    values = finite(values); width = min(4, len(values)); return (float(np.mean(values[:width])) if side == 'early' else float(np.mean(values[-width:]))) if width else None

comparison_rows = []; per_n_rows = []
for clip_norm in CLIP_VALUES:
    branch = branches[clip_norm]
    raw_training = [json.loads(line) for line in (branch['root'] / 'training_health.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    training_rows = list({int(row['logical_batch']): row for row in raw_training}.values())
    training_rows.sort(key=lambda row: int(row['logical_batch']))
    raw_trajectories = [json.loads(line) for line in (branch['root'] / 'trajectory_tb_diagnostics.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    trajectory_rows = list({(int(row['logical_batch']), int(row['candidate_index_in_batch'])): row for row in raw_trajectories}.values())
    trajectory_rows.sort(key=lambda row: (int(row['logical_batch']), int(row['candidate_index_in_batch'])))
    successful = [row for row in training_rows if row['successful_optimizer_update']]
    comparison_rows.append({'policy_max_norm': clip_norm, 'logical_batches': len(training_rows), 'successful_updates': len(successful), 'skipped_batches': len(training_rows)-len(successful), 'skip_rate': (len(training_rows)-len(successful))/len(training_rows), 'pre_clip_grad_norm_median': median([r['model_gradient_norm_before_clip'] for r in successful]), 'clip_coefficient_median': median([r['model_gradient_clip_coefficient'] for r in successful]), 'clipping_fraction': float(np.mean([r['model_gradient_clip_coefficient'] < 1.0 for r in successful])), 'parameter_update_norm_median': median([r['model_parameter_update_norm'] for r in successful]), 'relative_update_norm_median': median([r['model_relative_update_norm'] for r in successful]), 'tb_rms_early': phase_mean([r['tb_delta_rms'] for r in successful], 'early'), 'tb_rms_late': phase_mean([r['tb_delta_rms'] for r in successful], 'late'), 'loss_early': phase_mean([r['loss'] for r in successful], 'early'), 'loss_late': phase_mean([r['loss'] for r in successful], 'late'), 'policy_entropy_early': phase_mean([r['policy_entropy_mean'] for r in successful], 'early'), 'policy_entropy_late': phase_mean([r['policy_entropy_mean'] for r in successful], 'late'), 'wall_seconds': float(sum(r['batch_wall_seconds'] for r in training_rows)), 'cuda_peak_memory_bytes': int(max(r['cuda_peak_memory_bytes'] for r in training_rows)), 'nonfinite_health_values': int(sum(not math.isfinite(float(r[key])) for r in successful for key in ('loss','tb_delta_rms','model_gradient_norm_before_clip','model_parameter_update_norm','policy_entropy_mean')))})
    for node_count in range(1, 21):
        rows = [r for r in trajectory_rows if int(r['target_node_count']) == node_count]
        exposed = [r for r in rows if r['successful_gradient_exposure']]
        deltas = [r['tb_delta'] for r in exposed]
        per_n_rows.append({'policy_max_norm': clip_norm, 'N': node_count, 'valid_trajectory_count': len(rows), 'successful_gradient_exposure_count': len(exposed), 'delta_early_mean': phase_mean(deltas, 'early'), 'delta_late_mean': phase_mean(deltas, 'late'), 'delta_std': float(np.std(finite(deltas), ddof=0)) if deltas else None})
comparison = pd.DataFrame(comparison_rows).set_index('policy_max_norm')
per_n = pd.DataFrame(per_n_rows).set_index(['policy_max_norm','N'])
comparison.to_csv(RUN_ROOT / 'clip_comparison_summary.csv'); per_n.to_csv(RUN_ROOT / 'clip_comparison_per_N.csv')
summary = {'schema': 'factor_gfn.policy_clip_comparison_6_20.v1', 'decision_status': 'manual_review_required', 'selection_uses_reward_or_ic': False, 'only_changed_parameter': 'model_gradient_clip_norm', 'target_successful_updates': TARGET_SUCCESSFUL_UPDATES, 'comparison': comparison.reset_index().to_dict(orient='records'), 'per_N': per_n.reset_index().to_dict(orient='records'), 'initial_state_equality_verified': True}
(RUN_ROOT / 'clip_comparison_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
display(comparison); display(per_n)
for branch in branches.values(): branch['registry'].close()
print('POLICY_CLIP_COMPARISON_COMPLETE', flush=True)
print('把结果目录交给 Codex 分析：', RUN_ROOT, flush=True)